# Project 2 - test.py

Mark Khusid

In [1]:
import pandas as pd
import numpy as np

import pickle

from scipy.integrate import simpson

from sklearn.impute import KNNImputer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import cross_val_score

## Load Model from Pickle File

In [14]:
with open('svm_model.pkl', 'rb') as f:
    svm_model = pickle.load(f)

## Load Test Data into Matrix

In [15]:
df_test = pd.read_csv('test.csv', header=None)

In [16]:
test_matrix = df_test.values
#test_matrix

## Feature Extraction

### Define Feature Extractor Helper Functions

In [17]:
def compute_first_derivative(glucose_data, delta_t=5):
    return np.diff(glucose_data) / delta_t

In [18]:
def compute_second_derivative(glucose_values, delta_t=5):
    second_derivative = np.zeros(len(glucose_values) - 2)
    for i in range(1, len(glucose_values) - 1):
        second_derivative[i - 1] = \
            (glucose_values[i + 1] - 2 * glucose_values[i] + glucose_values[i - 1]) / (delta_t ** 2)
    return second_derivative

In [19]:
#first_derivative(cleaned_imputed_meal_matrix[0])

In [20]:
#second_derivative(cleaned_imputed_meal_matrix[0])

In [21]:
def compute_AUC(glucose_values, delta_t=5):
    return simpson(glucose_values, dx=delta_t)

### Define Feature Extractor Function

In [22]:
def feature_extractor(data_matrix):

    features = []

    for row in data_matrix:
        mean_glucose = np.mean(row)
        std_glucose  = np.std(row)
        max_glucose  = np.max(row)
        min_glucose  = np.min(row)
        
        first_derivative = compute_first_derivative(row)
        mean_first_derivate = np.mean(first_derivative)

        if len(row) > 2:
            second_derivative = compute_second_derivative(row)
            mean_second_derivative = np.mean(second_derivative)
        else:
            mean_second_derivative = 0

        AUC = compute_AUC(row)

        time_to_peak = np.argmax(row)

        features_vector = [
            mean_glucose,
            std_glucose,
            max_glucose,
            min_glucose,
            mean_first_derivate,
            mean_second_derivative,
            AUC,
            time_to_peak
        ]

        features.append(features_vector)
    
    return np.array(features)

### Get Feature Matrix for Test Data

In [23]:
test_features_matrix = \
    feature_extractor(test_matrix)

## Apply Model to Test Data

In [24]:
y_pred = svm_model.predict(test_features_matrix)

In [25]:
y_pred

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

## Save Predictions to CSV File

In [27]:
np.savetxt('Result.csv', y_pred.reshape(-1, 1), delimiter=',', fmt='%d')